# 🧑‍🔬 AI Scientist - Generate Your First Research Paper!

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SakanaAI/AI-Scientist/blob/main/notebooks/ai_scientist_2d_diffusion.ipynb)

Welcome! This notebook lets you run **The AI Scientist** on Google Colab to automatically:
1. Generate novel research ideas
2. Run experiments
3. Write a complete research paper
4. Create publication-ready figures

**Template**: 2D Diffusion (studies diffusion models on low-dimensional datasets)

---

### LLM Provider Comparison

| Provider | Model | Cost per Paper | Quality | Speed | Recommended For |
|----------|-------|----------------|---------|-------|-----------------|
| **DeepSeek** | deepseek-chat | **$1-2** | Good | Fast | Budget users |
| **OpenAI** | gpt-4o-mini | **$2-4** | Good | Fast | Most users |
| **OpenAI** | gpt-4o | **$5-8** | Great | Medium | Better quality |
| **Anthropic** | Claude Sonnet 3.5 | **$8-12** | **Best** | Medium | Best results |

### Cost Breakdown by Stage

| Stage | Time | API Calls | Cost (GPT-4o-mini) | Cost (Claude Sonnet) |
|-------|------|-----------|-------------------|---------------------|
| Setup | 5 min | 0 | Free | Free |
| Baseline Run | 5-10 min | 0 | Free | Free |
| Idea Generation | 2-5 min | ~5-10 | $0.50-1 | $1-2 |
| Experiments (Aider) | 10-20 min | ~20-50 | $1-3 | $4-8 |
| Paper Writing | 5-10 min | ~10-20 | $0.50-1 | $2-3 |
| **TOTAL** | **30-45 min** | **~35-80** | **$2-5** | **$7-13** |

*Note: Costs scale linearly with number of ideas. Above estimates are for 1 idea.*

### Cost Control Tips

1. **Start with 1 idea**: Default setting keeps costs low
2. **Use cheaper models**: DeepSeek or GPT-4o-mini for testing
3. **Skip novelty check**: Saves ~5-10 API calls per idea
4. **Skip review step**: Saves ~10-20 API calls (can review manually)
5. **Use free GPU**: Colab T4 is sufficient, no need for paid tiers



---
# Part 1: Setup & Installation 

Installing dependencies and configuring the environment.

In [ ]:
# Check GPU availability
import subprocess
import sys

gpu_info = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
if gpu_info.returncode == 0:
    print("GPU detected: ")
    print(gpu_info.stdout)
else:
    print("No GPU detected!")
    print("Please enable GPU: Runtime → Change runtime type → T4 GPU")
    sys.exit(1)

In [ ]:
%%bash
# Install minimal LaTeX (~500MB vs 5GB texlive-full)
echo "Installing LaTeX (this takes ~3-5 minutes)..."
apt-get update -qq
apt-get install -y -qq \
    texlive-latex-base \
    texlive-latex-extra \
    texlive-fonts-recommended \
    texlive-fonts-extra \
    cm-super \
    dvipng \
    chktex \
    > /dev/null 2>&1

echo "LaTeX installed successfully!"
pdflatex --version | head -1

In [ ]:
# Clone AI Scientist repository
import os

if not os.path.exists('AI-Scientist'):
    !git clone https://github.com/SakanaAI/AI-Scientist.git
    print("Repository cloned")
else:
    print("Repository already exists")

%cd AI-Scientist

In [ ]:
# Install Python dependencies
print("Installing Python dependencies...")
!pip install -q -r requirements.txt

# Install 2D Diffusion specific dependencies
!git clone https://github.com/gregversteeg/NPEET.git
%cd NPEET
!pip install -q .
%cd ..
!pip install -q scikit-learn

print("All dependencies installed!")

## Configure API Key

Choose **one** of these options:

### Option A: Use Colab Secrets (Recommended)
1. Click the 🔑 key icon in the left sidebar
2. Add a new secret:
   - Name: `OPENAI_API_KEY` (or `ANTHROPIC_API_KEY`, `DEEPSEEK_API_KEY`)
   - Value: Your API key
3. Enable "Notebook access"
4. Run the cell below

### Option B: Manual Entry
Uncomment and run the manual entry cell instead.

In [ ]:
# Option A: Use Colab Secrets
from google.colab import userdata
import os

# Try to get key from secrets 
try:
    os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')
    print("OpenAI API key loaded from secrets")
    model_choice = "gpt-4o-mini"  # Affordable and good quality 
except:
    try:
        os.environ['ANTHROPIC_API_KEY'] = userdata.get('ANTHROPIC_API_KEY')
        print("Anthropic API key loaded from secrets")
        model_choice = "claude-3-5-sonnet-20241022"  # Best quality
    except:
        try:
            os.environ['DEEPSEEK_API_KEY'] = userdata.get('DEEPSEEK_API_KEY')
            print("DeepSeek API key loaded from secrets")
            model_choice = "deepseek-chat"  # Cheapest 
        except:
            print("No API key found in secrets. Use Option B below.")
            model_choice = None

In [ ]:
# Option B: Manual Entry (uncomment to use)
import getpass
import os

print("Choose your LLM provider:")
print("1. OpenAI (gpt-4o-mini) - ~$2-4 per paper")
print("2. Anthropic (claude-3-5-sonnet) - ~$8-12 per paper")
print("3. DeepSeek (deepseek-chat) - ~$1-2 per paper")

provider = input("Enter choice (1/2/3): ").strip()

if provider == "1":
    os.environ['OPENAI_API_KEY'] = getpass.getpass('Enter your OpenAI API key: ')
    model_choice = "gpt-4o-mini"
elif provider == "2":
    os.environ['ANTHROPIC_API_KEY'] = getpass.getpass('Enter your Anthropic API key: ')
    model_choice = "claude-3-5-sonnet-20241022"
elif provider == "3":
    os.environ['DEEPSEEK_API_KEY'] = getpass.getpass('Enter your DeepSeek API key: ')
    model_choice = "deepseek-chat"
else:
    raise ValueError("Invalid choice")

print(f"\nUsing model: {model_choice}")

In [ ]:
# Verify setup
import torch

assert model_choice is not None, "No API key configured! Use Option A or B above."
assert torch.cuda.is_available(), "No GPU available! Change runtime type to GPU."

print("\n" + "="*50)
print("Setup Complete!")
print("="*50)
print(f"Model: {model_choice}")
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"Working directory: {os.getcwd()}")
print("="*50)

---
# Part 2: Generate Baseline Run 

Running the baseline experiment to establish performance metrics.

This trains a simple diffusion model on 2D datasets 

In [ ]:
import subprocess
import os

template_dir = "templates/2d_diffusion"
os.chdir(template_dir)

print("🔬 Running baseline experiment...")
print("This will train a diffusion model on 2D datasets.")
print("Expected time: 5-10 minutes\n")

# Run baseline
result = subprocess.run(
    ["python", "experiment.py", "--out_dir=run_0"],
    capture_output=False,  # Show output
    text=True
)

if result.returncode == 0:
    print("\nBaseline experiment completed!")
else:
    print("\nBaseline experiment failed!")

In [ ]:
# Display baseline results
import json
from IPython.display import Image, display
import os

# Show final metrics
with open("run_0/final_info.json", "r") as f:
    baseline_results = json.load(f)

print("Baseline Results:")
print("=" * 50)
for dataset, metrics in baseline_results.items():
    print(f"\n{dataset}:")
    for metric, value in metrics.items():
        print(f"  {metric}: {value}")

# Generate and show plots
print("\nGenerating plots...")
!python plot.py

# Display generated plots
plot_files = [f for f in os.listdir('.') if f.endswith('.png')]
print(f"\nGenerated {len(plot_files)} plots:\n")
for plot_file in sorted(plot_files)[:3]:  # Show first 3 plots
    print(f"Showing: {plot_file}")
    display(Image(filename=plot_file, width=400))

---
# Part 3: Generate Research Ideas

The AI will now generate novel research ideas based on the baseline results.

**Cost control**: Default to generating 1 idea. You can increase this, but it will be costly!

In [ ]:
# Configuration
NUM_IDEAS = 1  # ⚠️ You can increase this, but it will be costly!
SKIP_NOVELTY_CHECK = True  # Set False to check against existing literature. Slower, more API calls

print(f"Configuration:")
print(f"  Model: {model_choice}")
print(f"  Number of ideas: {NUM_IDEAS}")
print(f"  Novelty check: {'Disabled' if SKIP_NOVELTY_CHECK else 'Enabled'}")
print(f"\nEstimated cost for idea generation: ${NUM_IDEAS * 0.5:.2f} - ${NUM_IDEAS * 2:.2f}")

In [ ]:
# Generate ideas by calling function directly
from ai_scientist.generate_ideas import generate_ideas
from ai_scientist.llm import create_client

print("Generating research ideas...\n")

  # Create client
client, client_model = create_client(model_choice)

  # Generate ideas with controlled count
ideas = generate_ideas(
  base_dir="templates/2d_diffusion",
  client=client,
  model=client_model,
  skip_generation=False,
  max_num_generations=NUM_IDEAS,  
  num_reflections=3,
)

print(f"\nGenerated {len(ideas)} ideas!")

In [ ]:
# Display generated ideas
import json

with open("templates/2d_diffusion/ideas.json", "r") as f:
    ideas = json.load(f)

print("\n" + "="*70)
print("Generated Research Ideas")
print("="*70)

novel_ideas = [idea for idea in ideas if idea.get("novel", True)]

for i, idea in enumerate(novel_ideas, 1):
    print(f"\n{'='*70}")
    print(f"Idea {i}: {idea['Title']}")
    print(f"{'='*70}")
    print(f"Name: {idea['Name']}")
    print(f"\nExperiment Plan:\n{idea['Experiment']}")
    print(f"\nRatings:")
    print(f"  Interestingness: {idea.get('Interestingness', 'N/A')}/10")
    print(f"  Feasibility: {idea.get('Feasibility', 'N/A')}/10")
    print(f"  Novelty: {idea.get('Novelty', 'N/A')}/10")

print(f"\n{'='*70}")
print(f"Total novel ideas: {len(novel_ideas)}")
print(f"{'='*70}")

---
# Part 4: Run Experiments

The AI will now:
1. Modify the experiment code to implement the idea
2. Run multiple experiments
3. Collect and analyze results

**Note**: This uses Aider (AI pair programmer) to write code autonomously.

In [ ]:
# Add "novel" field to all ideas
import json

ideas_file = "templates/2d_diffusion/ideas.json"

# Load ideas
with open(ideas_file, "r") as f:
    ideas = json.load(f)

for idea in ideas:
    if "novel" not in idea:
        idea["novel"] = True


with open(ideas_file, "w") as f:
    json.dump(ideas, f, indent=4)

print(f"Fixed {len(ideas)} ideas — added 'novel' field")


In [ ]:
# Part 4: Run AI Scientist  

print("=" * 70)
print("Run FULL AI Scientist Pipeline")
print("=" * 70)

import os
import subprocess
import json

# Check environment
os.chdir("/content/AI-Scientist")
print("\n✓ Current directory:", os.getcwd())
print("✓ Ideas file:", "EXISTS" if os.path.exists("templates/2d_diffusion/ideas.json") else "MISSING")
print("✓ Launch script:", "EXISTS" if os.path.exists("launch_scientist.py") else "MISSING")

# Try to see what ideas we have
try:
    with open("templates/2d_diffusion/ideas.json", "r") as f:
        ideas = json.load(f)
    print(f"✓ Found {len(ideas)} ideas")
    print(f"✓ First idea: {ideas[0]['Name']}")
except Exception as e:
    print(f"✗ Error reading ideas: {e}")

print("\n" + "=" * 70)
print("RUNNING....")
print("=" * 70 + "\n")

# Run with FULL error capture
result = subprocess.run([
    "python", "launch_scientist.py",
    "--model", model_choice,
    "--experiment", "2d_diffusion",
    "--num-ideas", "1",
    "--skip-idea-generation",
    "--skip-novelty-check",
    "--parallel", "0"
], capture_output=True, text=True)

print("STDOUT:")
print(result.stdout)

print("\nSTDERR:")
print(result.stderr)

print("\nReturn code:", result.returncode)

if result.returncode != 0:
    print("\n" + "=" * 70)
    print("FAILED - See errors above")
    print("=" * 70)
else:
    print("\n" + "=" * 70)
    print("🎉 SUCCESS!")
    print("=" * 70)


### Part 5 — View and Download Generated Paper

This section locates the most recently generated research paper from the `2d_diffusion` experiment, displays it directly within the notebook, and provides download options for both the paper and the full experiment results.

#### What this cell does
1. Searches the results directory for generated `.pdf` files.  
2. Displays the most recent paper inline for quick review.  
3. Allows you to download either the paper alone or the entire results folder as a `.zip` archive.

#### If no PDF is found
- Confirm that the previous pipeline cell finished successfully.  
- Review the console output above for any runtime or file-generation errors.  

#### To generate a new paper manually
```bash
python launch_scientist.py --model {model_choice} --experiment 2d_diffusion --num-ideas 1
```
---

In [ ]:
# View and Download Generated Paper

import os
import glob
from IPython.display import IFrame, display
from google.colab import files

print("Looking for generated paper...")

results_dir = "/content/AI-Scientist/results/2d_diffusion"
pdf_files = glob.glob(f"{results_dir}/*/*.pdf")

if not pdf_files:
    print("No PDF found.")
    print(f"Checked directory: {results_dir}")
    print("\nTroubleshooting:")
    print("- Check if the pipeline completed successfully above.")
    print("- Look for error messages in the output.")
else:
    # Get the most recent PDF
    latest_pdf = max(pdf_files, key=os.path.getctime)
    paper_name = os.path.basename(latest_pdf)

    print(f"✓ Found paper: {paper_name}")
    print(f"Location: {latest_pdf}")
    print(f"Size: {os.path.getsize(latest_pdf) / 1024:.1f} KB")

    print("\n" + "=" * 70)
    print("Displaying generated paper")
    print("=" * 70 + "\n")

    display(IFrame(latest_pdf, width=900, height=1200))

    print("\n" + "=" * 70)
    print("Download options")
    print("=" * 70)

    # Download the PDF
    print("\nDownloading PDF to your computer...")
    files.download(latest_pdf)

    print("\nTo download the full results (code, plots, and paper), run this cell:")
    print(f"""
import shutil
shutil.make_archive('/content/full_results', 'zip', '{os.path.dirname(latest_pdf)}')
from google.colab import files
files.download('/content/full_results.zip')
""")

    print("\nProcess complete. The AI-generated paper is ready.")


---
# Run Complete — AI Scientist 
Congratulations! You've successfully run The AI Scientist to generate a complete research paper.
You’ve successfully:
- Installed and configured the AI Scientist framework.
- Validated your environment with the debug pipeline.
- Generated, visualized, and downloaded a full AI-authored research paper.

### Next recommended steps
- Explore other experiment templates in the `experiments/` folder.
- Modify existing templates or create your own custom experiments.

### Try other templates:
- `grokking` - Studies sudden generalization in neural networks
- `nanoGPT` - Experiments with small language models
- `probes` - LLM steering and interpretability

### Learn more:
- 📄 [Paper](https://arxiv.org/abs/2408.06292)
- 📝 [Blog Post](https://sakana.ai/ai-scientist/)
- 💻 [GitHub](https://github.com/SakanaAI/AI-Scientist)
- 📂 [Example Papers](https://drive.google.com/drive/folders/1G7A0wTqfXVa-cpexjk0oaXakaSJwffEt)

### Share your results!
- Tweet about your generated papers with #AIScientist
- Open issues on GitHub if you find bugs
- Contribute new templates!

---

**Questions or issues?** Check the [GitHub Issues](https://github.com/SakanaAI/AI-Scientist/issues) or [Discussion](https://github.com/SakanaAI/AI-Scientist/discussions)
